In [ ]:
import pandas as pd
import numpy as np

In [ ]:
movies = pd.read_csv('../data/raw/movie.csv')
ratings = pd.read_csv('../data/raw/rating.csv')

In [ ]:
# Get unique users
unique_users = ratings['userId'].unique()

# Set random seed for reproducibility and pick 5% of users
np.random.seed(42)
sampled_users = np.random.choice(
    unique_users, 
    size=int(len(unique_users) * 0.05), 
    replace=False
)

# Filter ratings to keep only sampled users
small_ratings = ratings[ratings['userId'].isin(sampled_users)].copy()

# Filter out noise (inactive users and unpopular movies)
USER_MIN_RATINGS = 20
MOVIE_MIN_RATINGS = 5

# Keep movies with at least 5 ratings
movie_counts = small_ratings['movieId'].value_counts()
popular_movies = movie_counts[movie_counts >= 5].index
cleaner_ratings = small_ratings[small_ratings['movieId'].isin(popular_movies)]

# Keep users with at least 20 ratings
user_counts = cleaner_ratings['userId'].value_counts()
active_users = user_counts[user_counts >= 20].index
filtered_ratings = cleaner_ratings[cleaner_ratings['userId'].isin(active_users)]

print(f"Final shape of ratings: {filtered_ratings.shape}")

In [ ]:
# Filter movies to match our cleaned ratings dataset
valid_movie_ids = filtered_ratings['movieId'].unique()
filtered_movies = movies[movies['movieId'].isin(valid_movie_ids)].copy()

# Process genres for LightFM item features
# Split the string 'Action|Adventure|Sci-Fi' into a list: ['Action', 'Adventure', 'Sci-Fi']
filtered_movies['genres'] = filtered_movies['genres'].str.split('|')

# Handle missing genres (MovieLens uses '(no genres listed)')
filtered_movies['genres'] = filtered_movies['genres'].apply(
    lambda x: [] if x == ['(no genres listed)'] else x
)

In [ ]:
filtered_ratings.head()

In [ ]:
filtered_movies.head()

In [ ]:
output_dir = '../data/preprocessed'

# Save the cleaned ratings
filtered_ratings.to_csv(f'{output_dir}/ratings_cleaned.csv', index=False)

# Save the cleaned movies and their features (genres)
filtered_movies.to_csv(f'{output_dir}/movies_cleaned.csv', index=False)